<div style="background: linear-gradient(135deg, #0f0f1a 0%, #1a1a2e 40%, #16213e 70%, #0f3460 100%); padding: 60px 40px; border-radius: 16px; border: 1px solid #e94560; box-shadow: 0 0 40px rgba(233,69,96,0.3); margin-bottom: 30px;">

<div style="text-align: center; margin-bottom: 20px;">
  <span style="background: #e94560; color: white; padding: 6px 18px; border-radius: 20px; font-size: 13px; font-weight: 600; letter-spacing: 2px; text-transform: uppercase;">COGNIFYZ TECHNOLOGIES · DATA SCIENCE INTERNSHIP</span>
</div>

<h1 style="color: #ffffff; text-align: center; font-size: 2.6em; font-weight: 800; letter-spacing: 1px; margin: 24px 0 10px; line-height: 1.2;">Geographic Analysis of Global Restaurant Distribution</h1>

<h3 style="color: #a8b2d8; text-align: center; font-weight: 400; font-size: 1.1em; margin-bottom: 30px; letter-spacing: 0.5px;">Spatial Pattern Recognition · Cluster Detection · Location Intelligence</h3>

<hr style="border: none; border-top: 1px solid #e9456040; margin: 20px 0;">

<div style="display: flex; justify-content: center; gap: 40px; flex-wrap: wrap; margin-top: 20px;">
  <div style="text-align: center;">
    <p style="color: #8892b0; font-size: 11px; text-transform: uppercase; letter-spacing: 2px; margin: 0;">Analyst</p>
    <p style="color: #ccd6f6; font-size: 15px; font-weight: 600; margin: 4px 0 0;">Karthikeyan Thirunavukkarasu</p>
  </div>
  <div style="text-align: center;">
    <p style="color: #8892b0; font-size: 11px; text-transform: uppercase; letter-spacing: 2px; margin: 0;">Organization</p>
    <p style="color: #ccd6f6; font-size: 15px; font-weight: 600; margin: 4px 0 0;">Cognifyz Technologies</p>
  </div>
  <div style="text-align: center;">
    <p style="color: #8892b0; font-size: 11px; text-transform: uppercase; letter-spacing: 2px; margin: 0;">Domain</p>
    <p style="color: #ccd6f6; font-size: 15px; font-weight: 600; margin: 4px 0 0;">Data Science & Analytics</p>
  </div>
</div>

</div>

---

## 1. Project Overview

This analysis performs a comprehensive **geographic investigation** of a global restaurant dataset. Using geospatial visualization and statistical clustering methods, the objective is to:

- **Map** all restaurant locations using their longitude and latitude coordinates across the globe
- **Identify** spatial clusters and high-density zones of restaurant concentration
- **Analyze** geographic patterns relative to city distribution, rating performance, and cuisine type
- **Derive** actionable location-intelligence insights relevant to market strategy

The dataset contains **9,551 restaurant records** spanning **15 countries** and **141 cities**.

---

## 2. Environment Setup & Library Imports

In [ ]:
# ─────────────────────────────────────────────────────────────────
# COGNIFYZ TECHNOLOGIES — Geographic Analysis
# Analyst: Karthikeyan Thirunavukkarasu
# ─────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap, Normalize
import matplotlib.ticker as ticker
import seaborn as sns

from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler
from scipy.spatial import ConvexHull
from scipy.stats import gaussian_kde

import warnings
warnings.filterwarnings('ignore')

# ── Global Aesthetic Configuration ──────────────────────────────
PALETTE = {
    'bg':        '#0f0f1a',
    'panel':     '#1a1a2e',
    'accent1':   '#e94560',
    'accent2':   '#0f3460',
    'accent3':   '#533483',
    'gold':      '#f5a623',
    'teal':      '#00b4d8',
    'text':      '#ccd6f6',
    'muted':     '#8892b0',
}

plt.rcParams.update({
    'figure.facecolor':  PALETTE['bg'],
    'axes.facecolor':    PALETTE['panel'],
    'axes.edgecolor':    '#2a2a4a',
    'axes.labelcolor':   PALETTE['text'],
    'axes.titlecolor':   PALETTE['text'],
    'xtick.color':       PALETTE['muted'],
    'ytick.color':       PALETTE['muted'],
    'text.color':        PALETTE['text'],
    'grid.color':        '#2a2a4a',
    'grid.linewidth':    0.6,
    'grid.alpha':        0.5,
    'font.family':       'DejaVu Sans',
    'axes.titlesize':    14,
    'axes.labelsize':    11,
    'legend.facecolor':  '#1a1a2e',
    'legend.edgecolor':  '#2a2a4a',
    'legend.fontsize':   9,
})

print("Libraries loaded successfully.")
print(f"Matplotlib   : {matplotlib.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"NumPy        : {np.__version__}")

## 3. Data Ingestion & Quality Assessment

In [ ]:
# ── Load Dataset ─────────────────────────────────────────────────
df = pd.read_csv('cognifyz_dataset.csv')

print("=" * 60)
print("  DATASET OVERVIEW")
print("=" * 60)
print(f"  Total records        : {len(df):,}")
print(f"  Total features       : {df.shape[1]}")
print(f"  Countries covered    : {df['Country Code'].nunique()}")
print(f"  Cities covered       : {df['City'].nunique()}")
print(f"  Latitude range       : {df['Latitude'].min():.4f} to {df['Latitude'].max():.4f}")
print(f"  Longitude range      : {df['Longitude'].min():.4f} to {df['Longitude'].max():.4f}")
print(f"  Missing coordinates  : {df[['Latitude','Longitude']].isnull().sum().sum()}")
print("=" * 60)

# Drop zero-coordinate anomalies
df_geo = df[(df['Latitude'] != 0) & (df['Longitude'] != 0)].copy()
print(f"  Records after geo filter : {len(df_geo):,}")
print()
df_geo.head(3)

In [ ]:
# ── Coordinate Summary Statistics ────────────────────────────────
geo_stats = df_geo[['Latitude', 'Longitude', 'Aggregate rating', 'Votes']].describe().round(4)
print("Coordinate & Rating Descriptive Statistics")
print("-" * 55)
print(geo_stats.to_string())

## 4. Global Restaurant Distribution Map

In [ ]:
# ── Figure 1: Global Scatter Map ─────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 11), facecolor=PALETTE['bg'])
ax.set_facecolor('#0d0d1a')

# Draw latitude/longitude grid lines
for lat in range(-90, 91, 30):
    ax.axhline(lat, color='#1e1e3a', linewidth=0.5, zorder=1)
for lon in range(-180, 181, 30):
    ax.axvline(lon, color='#1e1e3a', linewidth=0.5, zorder=1)

# Equator & Prime Meridian
ax.axhline(0,  color='#2a2a5a', linewidth=1.2, linestyle='--', zorder=2)
ax.axvline(0,  color='#2a2a5a', linewidth=1.2, linestyle='--', zorder=2)

# Tropic lines
ax.axhline(23.5,  color='#1c3a3a', linewidth=0.8, linestyle=':', zorder=2)
ax.axhline(-23.5, color='#1c3a3a', linewidth=0.8, linestyle=':', zorder=2)

# Rating-based color
norm = Normalize(vmin=df_geo['Aggregate rating'].min(), vmax=df_geo['Aggregate rating'].max())
cmap = LinearSegmentedColormap.from_list('rating_cmap', ['#e94560', '#f5a623', '#00b4d8', '#00ff88'])

sc = ax.scatter(
    df_geo['Longitude'], df_geo['Latitude'],
    c=df_geo['Aggregate rating'],
    cmap=cmap, norm=norm,
    s=12, alpha=0.75, edgecolors='none',
    zorder=5
)

# Colorbar
cbar = plt.colorbar(sc, ax=ax, orientation='vertical', fraction=0.018, pad=0.01)
cbar.set_label('Aggregate Rating', color=PALETTE['text'], fontsize=10)
cbar.ax.yaxis.set_tick_params(color=PALETTE['muted'])
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=PALETTE['muted'])

# Highlight dense India cluster
india_box = plt.Rectangle((68, 8), 29, 29, fill=False,
                           edgecolor=PALETTE['gold'], linewidth=1.5,
                           linestyle='--', zorder=6)
ax.add_patch(india_box)
ax.text(97.5, 18, 'India\nCluster', color=PALETTE['gold'],
        fontsize=8.5, fontweight='bold', va='center', zorder=7)

ax.set_xlim(-180, 180)
ax.set_ylim(-60, 75)
ax.set_xlabel('Longitude', fontsize=11, labelpad=8)
ax.set_ylabel('Latitude',  fontsize=11, labelpad=8)
ax.set_title('Global Restaurant Distribution — Colored by Aggregate Rating',
             fontsize=16, fontweight='bold', pad=18, color=PALETTE['text'])

# Annotation labels
for lon_a, lat_a, label in [(-100, -50, 'Americas'), (10, -45, 'Africa'),
                              (130, -30, 'Oceania'), (80, 60, 'Asia')]:
    ax.text(lon_a, lat_a, label, color='#3a3a6a', fontsize=9,
            fontstyle='italic', ha='center', zorder=4)

ax.text(0.01, 0.02, 'Cognifyz Technologies | Karthikeyan Thirunavukkarasu',
        transform=ax.transAxes, color=PALETTE['muted'],
        fontsize=8, va='bottom', style='italic')

plt.tight_layout()
plt.savefig('fig1_global_map.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['bg'])
plt.show()
print("Figure 1 saved: fig1_global_map.png")

## 5. Density Heatmap — Geographic Concentration

In [ ]:
# ── Figure 2: Kernel Density Estimation Map ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(22, 9), facecolor=PALETTE['bg'])

# ── Panel A: Global KDE ──────────────────────────────────────────
ax = axes[0]
ax.set_facecolor('#0d0d1a')

lons = df_geo['Longitude'].values
lats = df_geo['Latitude'].values

kde = gaussian_kde(np.vstack([lons, lats]), bw_method=0.08)
lon_grid = np.linspace(-180, 180, 400)
lat_grid = np.linspace(-60, 75,  300)
LON, LAT = np.meshgrid(lon_grid, lat_grid)
Z = kde(np.vstack([LON.ravel(), LAT.ravel()])).reshape(LON.shape)

heat_cmap = LinearSegmentedColormap.from_list(
    'heat', ['#0d0d1a', '#0f3460', '#533483', '#e94560', '#f5a623', '#ffffff'])

im = ax.contourf(LON, LAT, Z, levels=40, cmap=heat_cmap)
ax.scatter(lons, lats, s=1.5, c='white', alpha=0.08, edgecolors='none')

plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01, label='Density')
ax.set_xlim(-180, 180)
ax.set_ylim(-60, 75)
ax.set_xlabel('Longitude', fontsize=10)
ax.set_ylabel('Latitude',  fontsize=10)
ax.set_title('Global KDE Density Heatmap', fontsize=13, fontweight='bold', pad=12)
ax.grid(True, alpha=0.15)

# ── Panel B: India/South-Asia Zoom ───────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#0d0d1a')

india = df_geo[(df_geo['Latitude'].between(8, 37)) &
               (df_geo['Longitude'].between(68, 97))]

kde2 = gaussian_kde(np.vstack([india['Longitude'], india['Latitude']]), bw_method=0.12)
lon2 = np.linspace(68, 97, 300)
lat2 = np.linspace(8, 37,  300)
L2, A2 = np.meshgrid(lon2, lat2)
Z2 = kde2(np.vstack([L2.ravel(), A2.ravel()])).reshape(L2.shape)

im2 = ax2.contourf(L2, A2, Z2, levels=40, cmap=heat_cmap)
ax2.scatter(india['Longitude'], india['Latitude'],
            s=4, c=PALETTE['gold'], alpha=0.4, edgecolors='none')

plt.colorbar(im2, ax=ax2, fraction=0.02, pad=0.01, label='Density')

# City annotations for India
city_coords = {
    'New Delhi': (77.21, 28.61),
    'Gurgaon':   (77.03, 28.46),
    'Noida':     (77.31, 28.57),
    'Mumbai':    (72.88, 19.07),
}
for city, (clon, clat) in city_coords.items():
    ax2.annotate(city, (clon, clat),
                 xytext=(clon + 1.5, clat + 0.8),
                 color='white', fontsize=8, fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color='#f5a623', lw=1))

ax2.set_xlabel('Longitude', fontsize=10)
ax2.set_ylabel('Latitude',  fontsize=10)
ax2.set_title('South Asia Regional Zoom — NCR Cluster', fontsize=13, fontweight='bold', pad=12)
ax2.grid(True, alpha=0.15)

fig.suptitle('Kernel Density Estimation — Restaurant Geographic Concentration',
             fontsize=16, fontweight='bold', color=PALETTE['text'], y=1.01)

plt.tight_layout()
plt.savefig('fig2_density_heatmap.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['bg'])
plt.show()
print("Figure 2 saved: fig2_density_heatmap.png")

## 6. DBSCAN Spatial Clustering

In [ ]:
# ── DBSCAN Clustering ─────────────────────────────────────────────
coords = df_geo[['Latitude', 'Longitude']].values
coords_rad = np.radians(coords)

# eps = 0.3 degrees (~33 km), min_samples = 15
db = DBSCAN(eps=0.3, min_samples=15, metric='haversine').fit(coords_rad)
df_geo = df_geo.copy()
df_geo['cluster'] = db.labels_

n_clusters  = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
n_noise     = (db.labels_ == -1).sum()
noise_pct   = n_noise / len(df_geo) * 100

print("=" * 55)
print("  DBSCAN CLUSTERING RESULTS")
print("=" * 55)
print(f"  Distinct clusters detected : {n_clusters}")
print(f"  Noise / Outlier points     : {n_noise:,}  ({noise_pct:.1f}%)")
print()

cluster_stats = (df_geo[df_geo['cluster'] != -1]
                 .groupby('cluster')
                 .agg(
                     count=('Restaurant ID', 'count'),
                     avg_rating=('Aggregate rating', 'mean'),
                     avg_votes=('Votes', 'mean'),
                     center_lat=('Latitude', 'mean'),
                     center_lon=('Longitude', 'mean'),
                     top_city=('City', lambda x: x.mode()[0])
                 )
                 .sort_values('count', ascending=False)
                 .reset_index())

cluster_stats['avg_rating'] = cluster_stats['avg_rating'].round(2)
cluster_stats['avg_votes']  = cluster_stats['avg_votes'].round(0).astype(int)
cluster_stats['center_lat'] = cluster_stats['center_lat'].round(4)
cluster_stats['center_lon'] = cluster_stats['center_lon'].round(4)

print("  Top 15 Clusters by Restaurant Count")
print("-" * 55)
print(cluster_stats.head(15).to_string(index=False))

In [ ]:
# ── Figure 3: DBSCAN Cluster Map ─────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 11), facecolor=PALETTE['bg'])
ax.set_facecolor('#0d0d1a')

for lat in range(-90, 91, 30):
    ax.axhline(lat, color='#1a1a30', linewidth=0.5)
for lon in range(-180, 181, 30):
    ax.axvline(lon, color='#1a1a30', linewidth=0.5)

# Noise points
noise = df_geo[df_geo['cluster'] == -1]
ax.scatter(noise['Longitude'], noise['Latitude'],
           s=5, c='#2a2a4a', alpha=0.4, label='Outliers', zorder=3)

# Clustered points — top clusters get vivid colors
top_clusters = cluster_stats['cluster'].head(12).tolist()
cluster_colors = plt.cm.tab20(np.linspace(0, 1, len(top_clusters)))

for idx, cid in enumerate(top_clusters):
    sub = df_geo[df_geo['cluster'] == cid]
    ax.scatter(sub['Longitude'], sub['Latitude'],
               s=14, color=cluster_colors[idx],
               alpha=0.85, edgecolors='none', zorder=5)

# Remaining clusters
remaining = df_geo[(df_geo['cluster'] != -1) & (~df_geo['cluster'].isin(top_clusters))]
ax.scatter(remaining['Longitude'], remaining['Latitude'],
           s=10, c=PALETTE['teal'], alpha=0.5, edgecolors='none',
           label='Other clusters', zorder=4)

# Annotate top cluster centroids
for _, row in cluster_stats.head(5).iterrows():
    ax.annotate(
        f"#{int(row['cluster'])}  {row['top_city']}\n{row['count']:,} restaurants",
        xy=(row['center_lon'], row['center_lat']),
        xytext=(row['center_lon'] + 5, row['center_lat'] + 4),
        color='white', fontsize=8,
        arrowprops=dict(arrowstyle='->', color=PALETTE['gold'], lw=1.2),
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#1a1a2e',
                  edgecolor=PALETTE['gold'], alpha=0.9),
        zorder=9
    )

ax.set_xlim(-180, 180)
ax.set_ylim(-60, 75)
ax.set_xlabel('Longitude', fontsize=11)
ax.set_ylabel('Latitude',  fontsize=11)
ax.set_title(f'DBSCAN Spatial Clustering — {n_clusters} Clusters Detected  |  eps=0.3°, min_samples=15',
             fontsize=15, fontweight='bold', pad=16)

ax.text(0.01, 0.02, 'Cognifyz Technologies | Karthikeyan Thirunavukkarasu',
        transform=ax.transAxes, color=PALETTE['muted'], fontsize=8, style='italic')

plt.tight_layout()
plt.savefig('fig3_dbscan_clusters.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['bg'])
plt.show()
print("Figure 3 saved: fig3_dbscan_clusters.png")

## 7. City-Level Geographic Analysis

In [ ]:
# ── Figure 4: City Distribution & Rating Bubble Map ──────────────
city_agg = (df_geo.groupby('City')
            .agg(
                count=('Restaurant ID', 'count'),
                avg_rating=('Aggregate rating', 'mean'),
                avg_votes=('Votes', 'mean'),
                lat=('Latitude', 'mean'),
                lon=('Longitude', 'mean')
            )
            .reset_index()
            .sort_values('count', ascending=False))

top_cities = city_agg.head(30)

fig, ax = plt.subplots(figsize=(20, 11), facecolor=PALETTE['bg'])
ax.set_facecolor('#0d0d1a')

for lat in range(-90, 91, 30):
    ax.axhline(lat, color='#1a1a30', linewidth=0.5)
for lon in range(-180, 181, 30):
    ax.axvline(lon, color='#1a1a30', linewidth=0.5)

# Background all cities
ax.scatter(df_geo['Longitude'], df_geo['Latitude'],
           s=4, c='#2a2a5a', alpha=0.25, edgecolors='none', zorder=3)

# Bubble map: size = count, color = avg_rating
norm_r = Normalize(vmin=top_cities['avg_rating'].min(),
                   vmax=top_cities['avg_rating'].max())
cmap_r = LinearSegmentedColormap.from_list('r', ['#e94560', '#f5a623', '#00ff88'])

sc2 = ax.scatter(
    top_cities['lon'], top_cities['lat'],
    s=top_cities['count'] / 5,
    c=top_cities['avg_rating'], cmap=cmap_r, norm=norm_r,
    alpha=0.9, edgecolors='white', linewidths=0.5, zorder=7
)

plt.colorbar(sc2, ax=ax, fraction=0.018, pad=0.01,
             label='Average Rating')

for _, row in top_cities.head(10).iterrows():
    ax.annotate(
        f"{row['City']}\n({row['count']:,})",
        (row['lon'], row['lat']),
        textcoords='offset points', xytext=(8, 4),
        color='white', fontsize=7.5, fontweight='bold',
        zorder=10
    )

# Size legend
for sv, sl in [(200, '1,000'), (500, '2,500'), (1000, '5,000')]:
    ax.scatter([], [], s=sv/5, c='white', alpha=0.6, label=f'{sl} restaurants')
ax.legend(title='Bubble Size', title_fontsize=9,
          loc='lower left', framealpha=0.6)

ax.set_xlim(-180, 180)
ax.set_ylim(-60, 75)
ax.set_xlabel('Longitude', fontsize=11)
ax.set_ylabel('Latitude',  fontsize=11)
ax.set_title('City-Level Restaurant Bubble Map — Size: Count  |  Color: Avg. Rating  (Top 30 Cities)',
             fontsize=14, fontweight='bold', pad=16)

ax.text(0.01, 0.02, 'Cognifyz Technologies | Karthikeyan Thirunavukkarasu',
        transform=ax.transAxes, color=PALETTE['muted'], fontsize=8, style='italic')

plt.tight_layout()
plt.savefig('fig4_city_bubble_map.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['bg'])
plt.show()
print("Figure 4 saved: fig4_city_bubble_map.png")

## 8. Rating Distribution Across Geographic Zones

In [ ]:
# ── Figure 5: Quadrant Analysis — 4-panel deep dive ──────────────
fig = plt.figure(figsize=(22, 14), facecolor=PALETTE['bg'])
gs  = GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.30)

# ── Panel A: Top 20 Cities Bar Chart ─────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
top20 = city_agg.head(20)
colors_bar = [PALETTE['accent1'] if i < 3 else
              PALETTE['gold']    if i < 7 else
              PALETTE['teal']    for i in range(len(top20))]

bars = ax1.barh(top20['City'][::-1], top20['count'][::-1],
                color=colors_bar[::-1], edgecolor='none', height=0.75)

for bar, val in zip(bars, top20['count'][::-1]):
    ax1.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', fontsize=8, color=PALETTE['muted'])

ax1.set_xlabel('Number of Restaurants', fontsize=10)
ax1.set_title('Top 20 Cities by Restaurant Count', fontsize=12, fontweight='bold', pad=12)
ax1.set_xlim(0, top20['count'].max() * 1.18)
ax1.tick_params(axis='y', labelsize=8.5)
ax1.grid(axis='x', alpha=0.3)

# ── Panel B: Rating vs Votes Scatter (geographic coloring) ───────
ax2 = fig.add_subplot(gs[0, 1])

lat_bins = pd.cut(df_geo['Latitude'], bins=[-90, 0, 23.5, 45, 90],
                   labels=['Southern', 'Tropical', 'Subtropical', 'Northern'])
zone_colors = {'Southern': PALETTE['accent1'], 'Tropical': PALETTE['gold'],
               'Subtropical': PALETTE['teal'],  'Northern': PALETTE['accent3']}

for zone, zcolor in zone_colors.items():
    mask = lat_bins == zone
    ax2.scatter(df_geo.loc[mask, 'Aggregate rating'],
                df_geo.loc[mask, 'Votes'],
                s=8, c=zcolor, alpha=0.5, edgecolors='none', label=zone)

ax2.set_xlabel('Aggregate Rating', fontsize=10)
ax2.set_ylabel('Votes', fontsize=10)
ax2.set_title('Rating vs. Votes by Latitudinal Zone', fontsize=12, fontweight='bold', pad=12)
ax2.set_yscale('symlog')
ax2.legend(fontsize=8.5)
ax2.grid(alpha=0.25)

# ── Panel C: Avg Rating by Top 15 Cities ─────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
top15_r = city_agg.nlargest(15, 'count')[['City','avg_rating','count']]
top15_r = top15_r.sort_values('avg_rating', ascending=True)

c_vals = plt.cm.RdYlGn(Normalize()(top15_r['avg_rating'].values))
ax3.barh(top15_r['City'], top15_r['avg_rating'],
         color=c_vals, edgecolor='none', height=0.72)

for i, (_, row) in enumerate(top15_r.iterrows()):
    ax3.text(row['avg_rating'] + 0.02, i,
             f"{row['avg_rating']:.2f}", va='center', fontsize=8.5,
             color=PALETTE['text'])

ax3.set_xlabel('Average Aggregate Rating', fontsize=10)
ax3.set_title('Average Rating — Top 15 Cities by Count', fontsize=12, fontweight='bold', pad=12)
ax3.set_xlim(0, 5.2)
ax3.axvline(3.5, color=PALETTE['accent1'], linewidth=1.2,
            linestyle='--', alpha=0.7, label='3.5 threshold')
ax3.legend(fontsize=8)
ax3.tick_params(axis='y', labelsize=9)

# ── Panel D: Country-wise Restaurant Share ────────────────────────
ax4 = fig.add_subplot(gs[1, 1])

country_map = {
    1: 'India', 14: 'Australia', 30: 'Brazil', 37: 'Canada',
    94: 'Indonesia', 148: 'New Zealand', 162: 'Philippines',
    166: 'Qatar', 184: 'Singapore', 189: 'South Africa',
    191: 'Sri Lanka', 208: 'Turkey', 214: 'UAE',
    215: 'United Kingdom', 216: 'United States'
}
df_geo['Country'] = df_geo['Country Code'].map(country_map).fillna('Other')
country_counts = df_geo['Country'].value_counts().head(10)

wedge_colors = [PALETTE['accent1'], PALETTE['gold'], PALETTE['teal'],
                PALETTE['accent3'], '#00ff88', '#ff6b6b', '#4ecdc4',
                '#45b7d1', '#96ceb4', '#ffeaa7']

wedges, texts, autotexts = ax4.pie(
    country_counts.values,
    labels=country_counts.index,
    colors=wedge_colors,
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.82,
    wedgeprops=dict(edgecolor=PALETTE['bg'], linewidth=2)
)
for t in texts:     t.set_color(PALETTE['text']);  t.set_fontsize(9)
for at in autotexts: at.set_color('white');         at.set_fontsize(7.5)

ax4.set_title('Restaurant Share by Country (Top 10)', fontsize=12, fontweight='bold', pad=12)

fig.suptitle('Geographic Deep-Dive — City, Zone & Country Patterns',
             fontsize=17, fontweight='bold', color=PALETTE['text'], y=1.01)

plt.savefig('fig5_quadrant_analysis.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['bg'])
plt.show()
print("Figure 5 saved: fig5_quadrant_analysis.png")

## 9. Longitude & Latitude Distribution Profiles

In [ ]:
# ── Figure 6: Marginal Distributions ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 7), facecolor=PALETTE['bg'])

for ax, col, label, color in [
    (axes[0], 'Latitude',  'Latitude (°N/S)',  PALETTE['accent1']),
    (axes[1], 'Longitude', 'Longitude (°E/W)', PALETTE['teal'])
]:
    ax.set_facecolor(PALETTE['panel'])
    data = df_geo[col]
    counts, bin_edges = np.histogram(data, bins=120)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    ax.bar(bin_centers, counts, width=(bin_edges[1]-bin_edges[0])*0.85,
           color=color, alpha=0.7, edgecolor='none')

    # KDE overlay
    kde_mg = gaussian_kde(data, bw_method=0.1)
    x_kde  = np.linspace(data.min(), data.max(), 500)
    y_kde  = kde_mg(x_kde) * len(data) * (bin_edges[1] - bin_edges[0])
    ax.plot(x_kde, y_kde, color='white', linewidth=2, alpha=0.9)

    ax.axvline(data.median(), color=PALETTE['gold'],
               linewidth=1.5, linestyle='--', label=f'Median: {data.median():.2f}')
    ax.axvline(data.mean(),   color=PALETTE['accent1'],
               linewidth=1.5, linestyle=':',  label=f'Mean: {data.mean():.2f}')

    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f'{col} Distribution of All Restaurants', fontsize=13, fontweight='bold', pad=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.25)

fig.suptitle('Marginal Geographic Distributions — Latitude & Longitude',
             fontsize=15, fontweight='bold', color=PALETTE['text'], y=1.02)

plt.tight_layout()
plt.savefig('fig6_marginal_distributions.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['bg'])
plt.show()
print("Figure 6 saved: fig6_marginal_distributions.png")

## 10. Cluster Quality & Summary Statistics

In [ ]:
# ── Figure 7: Cluster Summary Heatmap ────────────────────────────
top_n = 15
cs = cluster_stats.head(top_n).set_index('top_city')
heatmap_data = cs[['count', 'avg_rating', 'avg_votes']].copy()
heatmap_data.columns = ['Restaurant Count', 'Avg. Rating', 'Avg. Votes']

# Normalize each column for heatmap
hm_norm = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min())

fig, ax = plt.subplots(figsize=(12, 8), facecolor=PALETTE['bg'])
ax.set_facecolor(PALETTE['panel'])

cmap_hm = LinearSegmentedColormap.from_list('hm', ['#1a1a2e', '#533483', '#e94560', '#f5a623'])
sns.heatmap(hm_norm, annot=heatmap_data, fmt='g', cmap=cmap_hm,
            linewidths=1.5, linecolor='#0f0f1a',
            annot_kws={'size': 9, 'color': 'white'},
            ax=ax, cbar_kws={'label': 'Normalized Score'})

ax.set_title(f'Cluster Quality Matrix — Top {top_n} Clusters by Restaurant Density',
             fontsize=13, fontweight='bold', pad=14, color=PALETTE['text'])
ax.set_xlabel('Metric', fontsize=10)
ax.set_ylabel('Dominant City', fontsize=10)
ax.tick_params(axis='x', labelsize=10, colors=PALETTE['text'])
ax.tick_params(axis='y', labelsize=9,  colors=PALETTE['text'])

plt.tight_layout()
plt.savefig('fig7_cluster_heatmap.png', dpi=150, bbox_inches='tight',
            facecolor=PALETTE['bg'])
plt.show()
print("Figure 7 saved: fig7_cluster_heatmap.png")

## 11. Key Findings & Analytical Summary

In [ ]:
# ── Print Structured Findings ─────────────────────────────────────
india_pct = round(len(df_geo[df_geo['Country'] == 'India']) / len(df_geo) * 100, 1)
ncr_count = len(df_geo[df_geo['City'].isin(['New Delhi', 'Gurgaon', 'Noida', 'Faridabad'])])
ncr_pct   = round(ncr_count / len(df_geo) * 100, 1)
top_cluster_city = cluster_stats.iloc[0]['top_city']
top_cluster_size = cluster_stats.iloc[0]['count']

print("\n" + "=" * 65)
print("  GEOGRAPHIC ANALYSIS — KEY FINDINGS")
print("  Cognifyz Technologies Data Science Internship")
print("  Analyst: Karthikeyan Thirunavukkarasu")
print("=" * 65)

print(f"""
1. DATASET SCOPE
   {len(df_geo):,} restaurants across {df_geo['Country'].nunique()} countries and {df_geo['City'].nunique()} cities.
   Coordinates span latitudes from {df_geo['Latitude'].min():.2f}° to {df_geo['Latitude'].max():.2f}°
   and longitudes from {df_geo['Longitude'].min():.2f}° to {df_geo['Longitude'].max():.2f}°.

2. DOMINANT GEOGRAPHIC CLUSTER — INDIA
   India accounts for {india_pct}% of all dataset restaurants.
   The NCR corridor (New Delhi / Gurgaon / Noida) alone hosts {ncr_count:,}
   restaurants — {ncr_pct}% of the entire dataset.

3. DBSCAN CLUSTERING
   {n_clusters} statistically significant spatial clusters identified.
   Largest cluster ({top_cluster_city}): {top_cluster_size:,} restaurants.
   {n_noise:,} points ({noise_pct:.1f}%) classified as geographic outliers.

4. LATITUDINAL BIAS
   Median latitude = {df_geo['Latitude'].median():.2f}° — strongly Northern Hemisphere skewed.
   The 25th–75th percentile latitude range is {df_geo['Latitude'].quantile(0.25):.2f}° to
   {df_geo['Latitude'].quantile(0.75):.2f}°, confirming concentration in subtropical Asia.

5. TOP RATED GEOGRAPHY
   Cities with the highest average rating among the top-15 by count:
   {city_agg.nlargest(15, 'count').nlargest(1, 'avg_rating')['City'].values[0]} with an average
   of {city_agg.nlargest(15, 'count').nlargest(1, 'avg_rating')['avg_rating'].values[0]:.2f} stars.

6. INTERNATIONAL PRESENCE
   Beyond India, significant clusters exist in the United States, UAE,
   United Kingdom, South Africa, Philippines, and Australia — indicating
   Zomato's international footprint is emerging but geographically
   dispersed relative to the South Asian core.
""")
print("=" * 65)

---

<div style="background: linear-gradient(135deg, #0f0f1a 0%, #1a1a2e 50%, #0f3460 100%); padding: 50px 40px; border-radius: 16px; border: 1px solid #e94560; box-shadow: 0 0 30px rgba(233,69,96,0.2); margin-top: 30px;">

<h2 style="color: #ffffff; text-align: center; font-size: 2em; font-weight: 700; margin-bottom: 10px;">Analysis Complete</h2>
<p style="color: #8892b0; text-align: center; font-size: 1em; margin-bottom: 30px;">Cognifyz Technologies — Data Science Internship</p>

<hr style="border: none; border-top: 1px solid #e9456030; margin: 20px 0;">

<div style="display: flex; justify-content: center; gap: 50px; flex-wrap: wrap;">
  <div style="text-align: center;">
    <p style="color: #8892b0; font-size: 11px; text-transform: uppercase; letter-spacing: 2px; margin: 0;">Analyst</p>
    <p style="color: #e94560; font-size: 16px; font-weight: 700; margin: 6px 0 0;">Karthikeyan Thirunavukkarasu</p>
  </div>
  <div style="text-align: center;">
    <p style="color: #8892b0; font-size: 11px; text-transform: uppercase; letter-spacing: 2px; margin: 0;">LinkedIn</p>
    <p style="color: #ccd6f6; font-size: 14px; margin: 6px 0 0;"><a href='https://www.linkedin.com/in/karthikeyan-thirunavukkarasu-2a2949305' style='color:#00b4d8; text-decoration:none;'>linkedin.com/in/karthikeyan-thirunavukkarasu</a></p>
  </div>
  <div style="text-align: center;">
    <p style="color: #8892b0; font-size: 11px; text-transform: uppercase; letter-spacing: 2px; margin: 0;">GitHub</p>
    <p style="color: #ccd6f6; font-size: 14px; margin: 6px 0 0;"><a href='https://github.com/karthiikofcl07' style='color:#00b4d8; text-decoration:none;'>github.com/karthiikofcl07</a></p>
  </div>
</div>

</div>